In [ ]:
import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd


# =========================
# 0. 기본 설정
# =========================

API_KEY = os.getenv("NEXON_API_KEY")
BASE_URL = "https://open.api.nexon.com/maplestory/v1"

HEADERS = {
    "x-nxopen-api-key": API_KEY
}

BASE_PATH = Path("/Users/john/Desktop/포폴_넥슨")
BASE_PATH.mkdir(parents=True, exist_ok=True)

DATE = "2026-06-04"

RANKING_MIN_LEVEL = 260
DETAIL_MIN_LEVEL = 285




In [ ]:
# =========================
# 1. 공통 API 호출
# =========================

def call_api(endpoint, params=None, max_retry=5, timeout=30):
    if params is None:
        params = {}

    url = BASE_URL + endpoint

    for attempt in range(max_retry):
        try:
            response = requests.get(
                url,
                headers=HEADERS,
                params=params,
                timeout=timeout
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code in [429, 500, 502, 503, 504]:
                wait_time = 2 ** attempt
                print(f"[RETRY] {endpoint}, status={response.status_code}, sleep={wait_time}")
                time.sleep(wait_time)
                continue

            print(f"[FAIL] {endpoint}, status={response.status_code}, params={params}, msg={response.text}")
            return None

        except requests.exceptions.RequestException as e:
            wait_time = 2 ** attempt
            print(f"[ERROR] {endpoint}, sleep={wait_time}, error={e}")
            time.sleep(wait_time)

    return None


In [ ]:
# =========================
# 2. 랭킹 260+ 수집
# =========================

def collect_ranking_dataframe(date, min_level=260):
    rows = []
    page = 1

    while True:
        data = call_api(
            endpoint="/ranking/overall",
            params={
                "date": date,
                "page": page
            }
        )

        if data is None:
            break

        ranking_list = data.get("ranking", [])

        if len(ranking_list) == 0:
            break

        stop = False

        for row in ranking_list:
            level = int(row.get("character_level", 0))

            if level >= min_level:
                rows.append({
                    "date": date,
                    "ranking": row.get("ranking"),
                    "world_name": row.get("world_name"),
                    "character_name": row.get("character_name"),
                    "character_level": level,
                    "character_exp": row.get("character_exp"),
                     "job_name": (
                         row.get("sub_class_name")
                         if row.get("sub_class_name")
                         else row.get("class_name")
                         )
                })
            else:
                stop = True

        print(f"[RANKING] page={page}")

        if stop:
            break

        if page % 1000 == 0:
            time.sleep(2)

        page += 1

    return pd.DataFrame(rows)

In [ ]:
# =========================
# 3. OCID 추출 - 285+ 대상
# =========================

def get_ocid(name):
    data = call_api(
        endpoint="/id",
        params={"character_name": name}
    )

    if data is None:
        return None

    return data.get("ocid")


def fetch_ocid_row(name):
    return {
        "character_name": name,
        "ocid": get_ocid(name)
    }


def build_ocid_dataframe(
    source_df,
    checkpoint_path,
    save_every=10000,
    max_workers=10
):
    checkpoint_path = Path(checkpoint_path)

    if checkpoint_path.exists():
        ocid_df = pd.read_csv(checkpoint_path)
        rows = ocid_df.to_dict("records")
        done_names = set(ocid_df["character_name"].dropna().astype(str).str.strip())
        print(f"[RESUME OCID] {len(done_names)}")
    else:
        rows = []
        done_names = set()

    names = (
        source_df["character_name"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    names = names[names != ""].drop_duplicates().tolist()

    target_names = [
        name for name in names
        if name not in done_names
    ]

    print(f"[OCID TARGET] {len(target_names)}")

    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(fetch_ocid_row, name): name
            for name in target_names
        }

        for future in as_completed(futures):
            name = futures[future]

            try:
                row = future.result()
            except Exception as e:
                row = {"character_name": name, "ocid": None}
                print(f"[OCID ERROR] {name}: {e}")

            rows.append(row)
            completed += 1

            if completed % 1000 == 0:
                print(f"[OCID] {completed}/{len(target_names)}")

            if len(rows) % save_every == 0:
                pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

    ocid_df = pd.DataFrame(rows)
    ocid_df.to_csv(checkpoint_path, index=False)

    return ocid_df

In [ ]:
# =========================
# 4. 심볼 포스 추출 - 285+ 대상
# =========================

def get_symbol_force_from_api(ocid, date):
    data = call_api(
        endpoint="/character/symbol-equipment",
        params={
            "ocid": ocid,
            "date": date
        }
    )

    if data is None:
        return {
            "ocid": ocid,
            "symbol_api_success": False,
            "arcane_symbol_force": None,
            "authentic_symbol_force": None
        }

    arcane_force = 0
    authentic_force = 0

    for symbol in data.get("symbol", []):
        name = symbol.get("symbol_name", "")
        force = int(symbol.get("symbol_force", 0))

        if name.startswith("아케인심볼"):
            arcane_force += force
        elif name.startswith("어센틱심볼") or name.startswith("그랜드 어센틱심볼"):
            authentic_force += force

    return {
        "ocid": ocid,
        "symbol_api_success": True,
        "arcane_symbol_force": arcane_force,
        "authentic_symbol_force": authentic_force
    }


def build_symbol_force_dataframe(
    source_df,
    date,
    checkpoint_path,
    save_every=10000,
    max_workers=10
):
    checkpoint_path = Path(checkpoint_path)

    if checkpoint_path.exists():
        symbol_df = pd.read_csv(checkpoint_path)
        rows = symbol_df.to_dict("records")
        done_ocids = set(symbol_df["ocid"].dropna().astype(str).str.strip())
        print(f"[RESUME SYMBOL] {len(done_ocids)}")
    else:
        rows = []
        done_ocids = set()

    ocids = source_df["ocid"].dropna().astype(str).str.strip()
    ocids = ocids[ocids != ""].drop_duplicates().tolist()

    target_ocids = [
        ocid for ocid in ocids
        if ocid not in done_ocids
    ]

    print(f"[SYMBOL TARGET] {len(target_ocids)}")

    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(get_symbol_force_from_api, ocid, date): ocid
            for ocid in target_ocids
        }

        for future in as_completed(futures):
            ocid = futures[future]

            try:
                row = future.result()
            except Exception as e:
                row = {
                    "ocid": ocid,
                    "symbol_api_success": False,
                    "arcane_symbol_force": None,
                    "authentic_symbol_force": None
                }
                print(f"[SYMBOL ERROR] {ocid}: {e}")

            rows.append(row)
            completed += 1

            if completed % 1000 == 0:
                print(f"[SYMBOL] {completed}/{len(target_ocids)}")

            if len(rows) % save_every == 0:
                pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

    symbol_df = pd.DataFrame(rows)
    symbol_df.to_csv(checkpoint_path, index=False)

    return symbol_df


In [ ]:
# =========================
# 5. HEXA 추출 - 285+ 대상
# =========================

def empty_hexa_row(ocid, api_success=False):
    default_level = 0 if api_success else None
    default_name = None

    row = {
        "ocid": ocid,
        "hexa_api_success": api_success,
        "has_hexa_core": None if not api_success else 0,
    }

    core_schema = {
        "mastery_core": 4,
        "enhance_core": 4,
        "skill_core": 6,
        "common_core": 4,   # 야누스 포함 + 추후 공용코어 대비
    }

    for core_prefix, max_count in core_schema.items():
        for i in range(1, max_count + 1):
            row[f"{core_prefix}_{i}_name"] = default_name
            row[f"{core_prefix}_{i}_level"] = default_level

    return row


def get_hexa_from_api(ocid, date):
    data = call_api(
        endpoint="/character/hexamatrix",
        params={
            "ocid": ocid,
            "date": date
        }
    )

    if data is None:
        return empty_hexa_row(ocid, api_success=False)

    hexa_list = data.get("character_hexa_core_equipment", [])

    base = empty_hexa_row(ocid, api_success=True)

    if len(hexa_list) > 0:
        base["has_hexa_core"] = 1

    counters = {
        "마스터리 코어": 0,
        "강화 코어": 0,
        "스킬 코어": 0,
        "공용 코어": 0,
    }

    prefix = {
        "마스터리 코어": "mastery_core",
        "강화 코어": "enhance_core",
        "스킬 코어": "skill_core",
        "공용 코어": "common_core",
    }

    for core in hexa_list:
        core_type = core.get("hexa_core_type")
        core_name = core.get("hexa_core_name")
        level = core.get("hexa_core_level")

        if core_type not in counters:
            continue

        counters[core_type] += 1

        col_prefix = f"{prefix[core_type]}_{counters[core_type]}"

        name_col = f"{col_prefix}_name"
        level_col = f"{col_prefix}_level"

        if name_col in base and level_col in base:
            base[name_col] = core_name
            base[level_col] = level

    return base

In [ ]:
# =========================
# 6. HEXA STAT 추출 - 285+ 대상
# =========================

def empty_hexa_stat_row(ocid, api_success=False):
    default_level = 0 if api_success else None
    default_name = None

    row = {
        "ocid": ocid,
        "hexa_stat_api_success": api_success,
        "has_hexa_stat": None if not api_success else 0,
    }

    for i in range(1, 4):
        row[f"stat_core_{i}_main_name"] = default_name
        row[f"stat_core_{i}_sub1_name"] = default_name
        row[f"stat_core_{i}_sub2_name"] = default_name

        row[f"stat_core_{i}_main_level"] = default_level
        row[f"stat_core_{i}_sub1_level"] = default_level
        row[f"stat_core_{i}_sub2_level"] = default_level

    return row


def get_hexa_stat_from_api(ocid, date):
    data = call_api(
        endpoint="/character/hexamatrix-stat",
        params={
            "ocid": ocid,
            "date": date
        }
    )

    if data is None:
        return empty_hexa_stat_row(ocid, api_success=False)

    base = empty_hexa_stat_row(ocid, api_success=True)

    core_keys = {
        1: "character_hexa_stat_core",
        2: "character_hexa_stat_core_2",
        3: "character_hexa_stat_core_3",
    }

    for i, key in core_keys.items():
        core_list = data.get(key, [])

        if not core_list:
            continue

        core = core_list[0]

        if core.get("main_stat_name") is not None:
            base["has_hexa_stat"] = 1

        base[f"stat_core_{i}_main_name"] = core.get("main_stat_name")
        base[f"stat_core_{i}_sub1_name"] = core.get("sub_stat_name_1")
        base[f"stat_core_{i}_sub2_name"] = core.get("sub_stat_name_2")

        base[f"stat_core_{i}_main_level"] = core.get("main_stat_level")
        base[f"stat_core_{i}_sub1_level"] = core.get("sub_stat_level_1")
        base[f"stat_core_{i}_sub2_level"] = core.get("sub_stat_level_2")

    return base


def build_hexa_stat_dataframe(
    source_df,
    date,
    checkpoint_path,
    save_every=10000,
    max_workers=10
):
    checkpoint_path = Path(checkpoint_path)

    if checkpoint_path.exists():
        hexa_stat_df = pd.read_csv(checkpoint_path)
        rows = hexa_stat_df.to_dict("records")
        done_ocids = set(
            hexa_stat_df["ocid"]
            .dropna()
            .astype(str)
            .str.strip()
        )
        print(f"[RESUME HEXA STAT] {len(done_ocids)}")
    else:
        rows = []
        done_ocids = set()

    ocids = source_df["ocid"].dropna().astype(str).str.strip()
    ocids = ocids[ocids != ""].drop_duplicates().tolist()

    target_ocids = [
        ocid for ocid in ocids
        if ocid not in done_ocids
    ]

    print(f"[HEXA STAT TARGET] {len(target_ocids)}")

    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(get_hexa_stat_from_api, ocid, date): ocid
            for ocid in target_ocids
        }

        for future in as_completed(futures):
            ocid = futures[future]

            try:
                row = future.result()
            except Exception as e:
                row = empty_hexa_stat_row(ocid, api_success=False)
                print(f"[HEXA STAT ERROR] {ocid}: {e}")

            rows.append(row)
            completed += 1

            if completed % 1000 == 0:
                print(f"[HEXA STAT] {completed}/{len(target_ocids)}")

            if len(rows) % save_every == 0:
                pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

    hexa_stat_df = pd.DataFrame(rows)
    hexa_stat_df.to_csv(checkpoint_path, index=False)

    return hexa_stat_df

In [ ]:
# =========================
# 7. 실행부
# =========================
BASE_PATH = Path("/Users/john/Desktop/포폴_넥슨")

DIRS = {
    "ranking": BASE_PATH / "ranking",
    "ocid": BASE_PATH / "ocid",
    "symbol": BASE_PATH / "symbol_force",
    "hexa": BASE_PATH / "hexa",
    "hexa_stat": BASE_PATH / "hexa_stat",
    "final": BASE_PATH / "final"
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

ranking_path = DIRS["ranking"] / f"ranking_260plus_{DATE}.csv"
detail_target_path = DIRS["ranking"] / f"detail_target_285plus_{DATE}.csv"

ocid_path = DIRS["ocid"] / f"ocid_285plus_{DATE}.csv"

symbol_path = DIRS["symbol"] / f"symbol_force_285plus_{DATE}.csv"

hexa_path = DIRS["hexa"] / f"hexa_285plus_{DATE}.csv"

hexa_stat_path = DIRS["hexa_stat"] / f"hexa_stat_285plus_{DATE}.csv"

final_path = DIRS["final"] / f"final_285plus_{DATE}.csv"



In [ ]:
# 1. 260+ 랭킹 수집
if ranking_path.exists():
    ranking_df = pd.read_csv(ranking_path)
else:
    ranking_df = collect_ranking_dataframe(
        date=DATE,
        min_level=RANKING_MIN_LEVEL
    )
    ranking_df.to_csv(ranking_path, index=False)

In [ ]:
# 2. 285+ 상세분석 대상 생성
detail_target_df = ranking_df[
    ranking_df["character_level"] >= DETAIL_MIN_LEVEL
].copy()

detail_target_df.to_csv(detail_target_path, index=False)

print(f"[RANKING 260+] {ranking_df.shape}")
print(f"[DETAIL 285+] {detail_target_df.shape}")

In [ ]:
# 3. OCID - 285+만
ocid_df = build_ocid_dataframe(
    source_df=detail_target_df,
    checkpoint_path=ocid_path,
    max_workers=10
)

detail_ocid_df = detail_target_df.merge(
    ocid_df,
    on="character_name",
    how="left"
)

In [ ]:
os.chdir("/Users/john/desktop/포폴_넥슨")

In [ ]:
# 4. 심볼 포스 - 285+만
symbol_df = build_symbol_force_dataframe(
    source_df=detail_ocid_df,
    date=DATE,
    checkpoint_path=symbol_path,
    max_workers=10
)

detail_symbol_df = detail_ocid_df.merge(
    symbol_df,
    on="ocid",
    how="left"
)


In [ ]:
# 5. HEXA - 285+만
hexa_df = build_hexa_dataframe(
    source_df=detail_symbol_df,
    date=DATE,
    checkpoint_path=hexa_path,
    max_workers=10
)

# 6. HEXA STAT - 285+만
hexa_stat_df = build_hexa_stat_dataframe(
    source_df=detail_symbol_df,
    date=DATE,
    checkpoint_path=hexa_stat_path,
    max_workers=10
)

final_df = detail_symbol_df.merge(
    hexa_df,
    on="ocid",
    how="left"
)

In [ ]:
final_df

In [ ]:
# 6. 최종 저장
final_df.to_csv(final_path, index=False)

print("[DONE]")
print(f"ranking_df: {ranking_df.shape}")
print(f"detail_target_df: {detail_target_df.shape}")
print(f"final_df: {final_df.shape}")